# Inspect RF features — sanity checks & engineered-feature sandbox

Companion to [rf_gapfill.py](rf_gapfill.py) and [features.py](features.py).

Two cheap paths:
1. **Single-slot grid** — `build_feature_grid(slot_utc)` for one timestamp. Fastest way to eyeball a new feature's value range on a (NLAT, NLON) grid.
2. **Small training table** — call `build_training_table` over a short window with a tiny `target_rows`. This is the distribution the RF actually *trains* on (post-stratification), which is what `df['fire_frp_log_24h'].describe()` should reflect.

Use Section 4 to scaffold a new engineered feature.

## 0 - Setup

In [ ]:
from __future__ import annotations
import os, sys
from datetime import date, datetime
from pathlib import Path

os.environ.setdefault('HDF5_USE_FILE_LOCKING', 'FALSE')
sys.path.insert(0, str(Path.cwd()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import config as cfg
import ancillary as anc
from features import build_feature_grid, stack_features, build_training_table
from slots    import iter_local_days, iter_window_slots

pd.set_option('display.float_format', lambda v: f'{v:.4f}')
print('RF_FEATURES:', cfg.RF_FEATURES)

## 1 - Single-slot grid (fast)

Pick a slot inside the training window and inspect every feature's per-cell statistics on the (NLAT, NLON) grid. No sampling, no I/O loop — one CAMS read, one ERA5 read, one IMERG window.

In [ ]:
# Pick the first window slot of the first training day that actually has
# Stage A coverage. iter_window_slots(day) is empty when discover_day_window
# returns None — common at the very edge of the training window.
probe_slot = None
for probe_day in iter_local_days(cfg.TRAIN_START, cfg.TRAIN_END):
    probe_slot = next(iter_window_slots(probe_day), None)
    if probe_slot is not None:
        break
if probe_slot is None:
    raise RuntimeError(
        f'No Stage A slots discoverable in [{cfg.TRAIN_START}, {cfg.TRAIN_END}].'
    )
print('probe day  (local):', probe_day.isoformat())
print('probe slot (UTC)  :', probe_slot.isoformat())

grids = build_feature_grid(probe_slot)

grid_stats = pd.DataFrame({
    name: pd.Series(g.ravel()).describe()
    for name, g in grids.items() if name in cfg.RF_FEATURES
}).T
grid_stats

In [ ]:
# Eyeball one feature as a map — defaults to fire_frp_log_24h.
FEATURE_TO_PLOT = 'fire_frp_log_24h'

g = grids[FEATURE_TO_PLOT]
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(g, origin='lower',
               extent=(cfg.LONS[0], cfg.LONS[-1], cfg.LATS[0], cfg.LATS[-1]),
               aspect='auto')
ax.set_title(f'{FEATURE_TO_PLOT}  @  {probe_slot.isoformat()}')
ax.set_xlabel('lon'); ax.set_ylabel('lat')
plt.colorbar(im, ax=ax); plt.show()

## 2 - Small training-table sample

Uses a *short* window and a tiny `target_rows` so this is minutes, not hours. The returned X is the matrix the RF sees post-stratification.

In [ ]:
# Quick sample — keep date range and target_rows small for an interactive run.
SAMPLE_START = cfg.TRAIN_START
SAMPLE_END   = cfg.TRAIN_START  # one day; widen if you have time.
TARGET_ROWS  = 20_000           # vs. cfg.RF_TRAIN_TARGET_ROWS in production.

X, y, meta = build_training_table(
    start=SAMPLE_START, end=SAMPLE_END,
    target_rows=TARGET_ROWS, progress=tqdm,
)
df = pd.concat([X, y.rename('aod_slot'), meta], axis=1)
print(f'rows: {len(df):,}   features: {len(X.columns)}')
df.head()

## 3 - The thing you asked for

In [ ]:
df['fire_frp_log_24h'].describe()

In [ ]:
# Zero-share is informative: log1p(0) == 0 for cells without a nearby fire.
zero_frac = float((df['fire_frp_log_24h'] == 0).mean())
nonzero_frac = float((df['fire_frp_log_24h'] > 0).mean())
nan_frac  = float(df['fire_frp_log_24h'].isna().mean())
print(f'fire_frp_log_24h: zero={zero_frac:.3f}  nonzero={nonzero_frac:.3f}  nan={nan_frac:.3f}')

fig, ax = plt.subplots(figsize=(6, 3))
df.loc[df['fire_frp_log_24h'] > 0, 'fire_frp_log_24h'].hist(bins=60, ax=ax)
ax.set_title('fire_frp_log_24h  (nonzero rows only)')
ax.set_xlabel('log1p(Σ FRP within radius, last 24 h)')
plt.show()

In [ ]:
# Per-feature describe across the sampled training rows.
X.describe().T

## 4 - Sandbox: try a new engineered feature

Build the candidate feature **outside** of `features.py` first so you can iterate without touching the production builder. Once it looks sane, lift it into `build_feature_grid` and add the name to `cfg.RF_FEATURES_DYNAMIC` / `_STATIC` / `_QUASISTATIC`.

In [ ]:
# TODO: define a candidate engineered feature on the per-slot grid.
#
# Contract:
#   build_candidate(grids: dict[str, np.ndarray], slot_utc: datetime) -> np.ndarray
#   - shape  : (cfg.NLAT, cfg.NLON)
#   - dtype  : float32
#   - NaN policy: NaN is OK (RF imputes via median); but constants/zeros
#     are usually a better default for proxies where 'absent' has meaning
#     (compare fire_frp_log_24h, which uses 0.0 not NaN).
#
# Some directions to consider (pick one — keep the implementation tight):
#   - 'wind_speed'         = sqrt(u10**2 + v10**2)
#   - 'wind_dir_sin/cos'   = sin/cos(atan2(v10, u10))   (cyclical encoding)
#   - 'tcwv_x_blh'         = tcwv * blh   (moisture-loaded boundary layer proxy)
#   - 'cams_minus_lag'     = cams_aod - cams_aod_lag_3h  (transport tendency)
#   - 'dry_spell_x_ssrd'   = hours_since_rain_0p1mm * ssrd  (photo-oxidation aging proxy)
#
# Trade-offs to weigh:
#   - Interaction terms (products) help linear-ish models; RF can often discover
#     them on its own but a shallow forest may not. Worth testing.
#   - Cyclical encodings only help when the raw direction wraps (wind, hour).
#   - Lag differences add a new noise floor — check the SD against the source
#     features' SDs before trusting it as signal.

CANDIDATE_NAME = 'TODO_new_feature'

def build_candidate(grids: dict, slot_utc: datetime) -> np.ndarray:
    # TODO: implement and return a (NLAT, NLON) float32 grid.
    raise NotImplementedError(
        'Define build_candidate(grids, slot_utc) — see TODO above.'
    )

# Probe on the single slot from Section 1.
cand_grid = build_candidate(grids, probe_slot)
assert cand_grid.shape == (cfg.NLAT, cfg.NLON), cand_grid.shape
assert cand_grid.dtype == np.float32, cand_grid.dtype

pd.Series(cand_grid.ravel()).describe()

In [ ]:
# Map view of the candidate.
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cand_grid, origin='lower',
               extent=(cfg.LONS[0], cfg.LONS[-1], cfg.LATS[0], cfg.LATS[-1]),
               aspect='auto')
ax.set_title(f'{CANDIDATE_NAME}  @  {probe_slot.isoformat()}')
ax.set_xlabel('lon'); ax.set_ylabel('lat')
plt.colorbar(im, ax=ax); plt.show()

In [ ]:
# Correlation of the candidate against the existing feature set, evaluated on
# the sampled training rows. The cheap proxy uses (row, col) from `meta` to
# look up the candidate value on each slot's grid — only practical when the
# sample window is short (we rebuild the grid for every distinct slot).
candidate_vals = np.empty(len(df), dtype=np.float32)
for (d, s_idx), g in df.groupby(['date', 'slot_idx']).groups.items():
    slot_utc = datetime(d.year, d.month, d.day, s_idx // 2, (s_idx % 2) * 30)
    g_grids  = build_feature_grid(slot_utc)
    c_grid   = build_candidate(g_grids, slot_utc)
    rows = df.loc[g, 'row'].to_numpy()
    cols = df.loc[g, 'col'].to_numpy()
    candidate_vals[g] = c_grid[rows, cols]

df_cand = X.copy()
df_cand[CANDIDATE_NAME] = candidate_vals
df_cand['aod_slot']     = y.to_numpy()

corr = df_cand.corr(numeric_only=True)[CANDIDATE_NAME].sort_values(ascending=False)
corr